# 2.1 Что такое AI-агент — практика

In [1]:
pip install langchain-openai langgraph

^C
Note: you may need to restart the kernel to use updated packages.


In [1]:
BASE_URL   = "http://localhost:11434/v1"
API_KEY    = "ollama"
MODEL_NAME = "qwen3:4b" # выберите ту LLM - которая поддерживает Function Calling!

---
## 1. ReAct-агент (LangGraph)

**ReAct** (Reason + Act) — паттерн, где LLM на каждом шаге:
1. **Думает** — нужен ли инструмент?
2. **Действует** — вызывает tool
3. **Наблюдает** — получает результат
4. Повторяет, пока не готов дать финальный ответ

В LangGraph всё это делает одна функция — `create_react_agent`.

In [2]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

# Подключаемся к LLM
model = ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, model=MODEL_NAME)

# Определяем инструмент — обычная Python-функция с декоратором @tool
@tool
def get_weather(city: str) -> str:
    """Возвращает текущую погоду в указанном городе."""
    data = {
        "москва": "облачно, +18°C",
        "лондон": "дождь, +12°C",
        "токио": "солнечно, +25°C",
    }
    return data.get(city.lower(), f"нет данных для {city}")


# Можете создать еще одну функцию и добавить агенту =)

demo_agent = create_react_agent(model, [get_weather])

C:\Users\Пользователь\AppData\Local\Temp\ipykernel_25472\1841751638.py:22: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  demo_agent = create_react_agent(model, [get_weather])


In [3]:
# Запускаем агента
result = demo_agent.invoke({"messages": [("user", "Какая погода в Москве?")]})

# Смотрим всю цепочку сообщений
for msg in result["messages"]:
    if msg.type == "human":
        print(f"User: {msg.content}")
    elif msg.type == "ai" and msg.tool_calls:
        calls = [f"{tc['name']}({tc['args']})" for tc in msg.tool_calls]
        print(f"AI вызывает: {', '.join(calls)}")
    elif msg.type == "tool":
        print(f"Tool [{msg.name}]: {msg.content}")
    elif msg.type == "ai":
        print(f"AI: {msg.content}")

User: Какая погода в Москве?
AI вызывает: get_weather({'city': 'Москва'})
Tool [get_weather]: облачно, +18°C
AI: В Москве сейчас погода облачно, температура +18°C.


**Что произошло:**
1. Мы передали вопрос агенту
2. LLM **сама решила** вызвать `get_weather` с аргументом `москва`
3. LangGraph выполнил функцию и вернул результат модели
4. Модель сформировала финальный ответ

Весь цикл Reason → Act → Observe → Answer — автоматически.

---
## 2. Задание: соберите своего агента

### Сценарий

Вы делаете агента-консультанта для магазина электроники.  
Клиент спрашивает про товар — агент должен найти информацию и проверить наличие.

### Что нужно сделать

1. Написать **два инструмента** (функции с `@tool`):
   - `search_product(query: str) -> str` — ищет товар в каталоге, возвращает название, цену, характеристики
   - `check_stock(product: str, city: str) -> str` — проверяет наличие товара в городе

2. Написать **системный промпт** — кто этот агент и как он должен себя вести

3. Создать агента через `create_react_agent` (смотри пример выше) и сохранить в переменную **`agent`**

Данные (словари-заглушки) уже готовы ниже. Ваша задача — написать функции и собрать агента.

In [4]:
# ── Данные (не менять) ────────────────────────────────────────

CATALOG = {
    "iphone 15": {"price": 999, "category": "смартфон", "specs": "6.1 дюйм, A16 Bionic, 128GB"},
    "samsung s24": {"price": 899, "category": "смартфон", "specs": "6.2 дюйм, Snapdragon 8 Gen 3, 128GB"},
    "macbook air": {"price": 1299, "category": "ноутбук", "specs": "13.6 дюйм, Apple M3, 8GB RAM"},
    "dell xps 15": {"price": 1199, "category": "ноутбук", "specs": "15.6 дюйм, Intel i7, 16GB RAM"},
}

STOCK = {
    "iphone 15": {"москва": 5, "санкт-петербург": 2},
    "samsung s24": {"москва": 0, "санкт-петербург": 8},
    "macbook air": {"москва": 3, "санкт-петербург": 0},
    "dell xps 15": {"москва": 1, "санкт-петербург": 4},
}

# ── ВАШ КОД ──────────────────────────────────────────────────

# 1. Инструмент: поиск товара
#    Функция должна найти товар в CATALOG по query (приведите к lower())
#    и вернуть строку с названием, ценой и характеристиками.
#    Если товар не найден — верните "Товар не найден".

# @tool
# def search_product(query: str) -> str:
#     """..."""
#     ...


# 2. Инструмент: проверка наличия
#    Функция должна проверить STOCK[product][city] и вернуть
#    количество в наличии или "нет в наличии".

# @tool
# def check_stock(product: str, city: str) -> str:
#     """..."""
#     ...


# 3. Системный промпт и агент
#    Подсказка: create_react_agent(model, tools, prompt=...)
#    prompt — это строка с системным промптом (необязательный параметр)

# agent = create_react_agent(model, [...], prompt="...")





# ── ВАШ КОД ──────────────────────────────────────────────────

@tool
def search_product(query: str) -> str:
    """Ищет товар в каталоге по названию и возвращает его цену и характеристики.
    Используй эту функцию, когда пользователь спрашивает про конкретный товар
    (например, сколько он стоит или какие у него характеристики)."""

    # приводим запрос к нижнему регистру — так же, как в get_weather с city.lower(),
    # чтобы "iPhone 15" и "iphone 15" находились одинаково
    query = query.lower()

    # ищем товар в словаре CATALOG по точному совпадению ключа
    product = CATALOG.get(query)

    # если товара нет — возвращаем текстовое сообщение об этом,
    # а не бросаем исключение: агент должен понять из текста, что делать дальше
    if product is None:
        return "Товар не найден"

    # формируем читаемую строку с данными о товаре
    return (
        f"{query} — {product['price']}$, "
        f"категория: {product['category']}, "
        f"характеристики: {product['specs']}"
    )


@tool
def check_stock(product: str, city: str) -> str:
    """Проверяет наличие товара на складе в конкретном городе и возвращает количество.
    Используй эту функцию, когда пользователь спрашивает, есть ли товар в наличии
    или сколько единиц товара доступно в конкретном городе."""

    # у этой функции два аргумента — LLM сама разберёт из запроса пользователя,
    # что подставить в product, а что в city, ориентируясь на docstring и их названия
    product = product.lower()
    city = city.lower()

    # сначала проверяем, что товар вообще существует в STOCK
    if product not in STOCK:
        return "Товар не найден"

    # затем проверяем наличие данных по конкретному городу для этого товара
    stock_by_city = STOCK[product]
    if city not in stock_by_city:
        return f"Нет данных по городу {city}"

    quantity = stock_by_city[city]

    # 0 в наличии — это не "нет данных", а именно "нет в наличии" по условию задания
    if quantity == 0:
        return "нет в наличии"

    return f"в наличии: {quantity} шт."


# ── 3. Системный промпт и агент ────────────────────────────────

# Системный промпт нужен, чтобы явно объяснить агенту, КАК пользоваться инструментами
# вместе — сам по себе он не знает, что для полного ответа про товар
# нужно вызвать ОБА инструмента (и search_product, и check_stock).
# Без этой подсказки модель может ограничиться только одним вызовом.
system_prompt = (
    "Ты — консультант интернет-магазина электроники. "
    "Когда пользователь спрашивает про товар (цену, характеристики или наличие), "
    "используй инструмент search_product, чтобы найти цену и характеристики, "
    "и инструмент check_stock, чтобы проверить наличие в конкретном городе. "
    "Если пользователь не указал город явно, уточни у него город или используй Москву по умолчанию. "
    "Отвечай развёрнуто на русском языке, объединяя данные из обоих инструментов в один связный ответ."
)

# create_react_agent собирает ReAct-цикл "из коробки":
# модель сама решает, какие из переданных инструментов вызывать и в каком порядке,
# опираясь на system_prompt и docstring'и самих инструментов (@tool)
agent = create_react_agent(
    model,
    [search_product, check_stock],  # список инструментов, доступных агенту
    prompt=system_prompt,
)

C:\Users\Пользователь\AppData\Local\Temp\ipykernel_25472\3401272234.py:125: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [5]:
# ── Проверка (запустите эту ячейку после написания кода) ──────

from langchain_core.messages import ToolMessage, AIMessage

print("Проверяем вашего агента...\n")
passed = 0
total = 4

# 1. Переменная agent существует
try:
    agent
    print("[1/4] agent определён")
    passed += 1
except NameError:
    print("[1/4] FAIL — переменная `agent` не определена")
    print("\nОстальные проверки пропущены. Создайте агента и попробуйте снова.")
    raise SystemExit

# 2. Агент запускается без ошибок
test_query = "Сколько стоит iPhone 15 и есть ли он в наличии в Москве?"
try:
    result = agent.invoke({"messages": [("user", test_query)]})
    print("[2/4] agent.invoke() отработал без ошибок")
    passed += 1
except Exception as e:
    print(f"[2/4] FAIL — ошибка при вызове: {e}")
    print("\nОстальные проверки пропущены. Исправьте ошибку и попробуйте снова.")
    raise SystemExit

# 3. Агент использовал минимум 2 разных инструмента
messages = result["messages"]
tool_names_used = set()
for m in messages:
    if isinstance(m, AIMessage) and m.tool_calls:
        for tc in m.tool_calls:
            tool_names_used.add(tc["name"])

if len(tool_names_used) >= 2:
    print(f"[3/4] Агент вызвал {len(tool_names_used)} инструмента: {tool_names_used}")
    passed += 1
else:
    print(f"[3/4] FAIL — агент вызвал {len(tool_names_used)} инструмент(ов): {tool_names_used}")
    print("       Нужно минимум 2. Проверьте промпт — агент должен и искать товар, и проверять наличие.")

# 4. Финальный ответ не пустой
final_msg = messages[-1]
if isinstance(final_msg, AIMessage) and final_msg.content and len(final_msg.content) > 10:
    print(f"[4/4] Финальный ответ получен ({len(final_msg.content)} символов)")
    passed += 1
else:
    print("[4/4] FAIL — агент не вернул финальный текстовый ответ")

print(f"\n{'=' * 40}")
if passed == total:
    print(f"Все проверки пройдены ({passed}/{total})!")
    print(f"\nОтвет агента:\n{final_msg.content}")
else:
    print(f"Пройдено {passed}/{total}. Исправьте ошибки и запустите проверку снова.")

Проверяем вашего агента...

[1/4] agent определён
[2/4] agent.invoke() отработал без ошибок
[3/4] Агент вызвал 2 инструмента: {'check_stock', 'search_product'}
[4/4] Финальный ответ получен (171 символов)

Все проверки пройдены (4/4)!

Ответ агента:
iPhone 15 стоит **999 долларов**. Это смартфон с экраном 6,1 дюйма, процессором A16 Bionic и объемом памяти 128 ГБ. В Москве на текущий момент в наличии **5 штук** товара.
